# 🎨 TryHairStyle — Full Training Pipeline

Pipeline huấn luyện **Deep Texture Hair Inpainting Model**.

## 📋 Pipeline
| Stage | Mô tả | Thời gian |
|-------|-------|----------|
| **Setup** | Clone code + tải SDXL model tự động | ~10 phút |
| **Stage 1** | Texture Encoder (ResNet50 + SupCon Loss) | ~30 phút |
| **Stage 2** | UNet Inpainting (9ch SDXL, Chunked Loading) | ~12-48h |
| **Stage 3** | Đánh giá & Export model | ~5 phút |

## 📦 Bạn chỉ cần upload
```
MyDrive/TryHairStyle/processed_001/
MyDrive/TryHairStyle/processed_002/
...
```
**Code** tự clone từ GitHub. **SDXL model** tự tải từ HuggingFace.

---
# PHẦN 1: SETUP TỰ ĐỘNG
---

## ⚙️ 1.1 — Kiểm tra GPU

In [ ]:
!nvidia-smi
import torch
print(f"\n🔥 PyTorch: {torch.__version__}")
print(f"🔥 CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"🔥 GPU: {torch.cuda.get_device_name(0)}")
    vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"🔥 VRAM: {vram:.1f} GB")
    if vram < 14:
        print("⚠️ VRAM < 14GB — Stage 2 có thể OOM. Nên dùng T4/V100/A100.")

## 📁 1.2 — Mount Drive + Cấu hình

Bạn chỉ cần upload các thư mục `processed_NNN/` lên Drive:
```
MyDrive/TryHairStyle/processed_001/
MyDrive/TryHairStyle/processed_002/
...
```

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

# ╔══════════════════════════════════════════════════════════╗
# ║  🔧 CẤU HÌNH — THAY ĐỔI NẾU CẦN                      ║
# ╚══════════════════════════════════════════════════════════╝
GITHUB_REPO = 'https://github.com/Phatjhhoq8/TryHairstyle_Diffusion.git'
GITHUB_BRANCH = 'master'

# Nơi chứa dataset chunks trên Drive
DRIVE_DATA = '/content/drive/MyDrive/TryHairStyle'
# Nơi lưu checkpoints trên Drive (tự tạo)
DRIVE_CHECKPOINTS = '/content/drive/MyDrive/TryHairStyle/checkpoints'
# ══════════════════════════════════════════════════════════

os.makedirs(DRIVE_CHECKPOINTS, exist_ok=True)
print(f"✅ Drive data: {DRIVE_DATA}")
print(f"✅ Checkpoints: {DRIVE_CHECKPOINTS}")

## 📥 1.3 — Clone code từ GitHub

In [ ]:
import os

LOCAL_PROJECT = '/content/TryHairStyle'

if os.path.exists(LOCAL_PROJECT):
    print("⏭️ Project đã có. Pull latest...")
    !cd {LOCAL_PROJECT} && git pull origin {GITHUB_BRANCH}
else:
    print(f"📥 Cloning {GITHUB_REPO}...")
    !git clone -b {GITHUB_BRANCH} --depth 1 {GITHUB_REPO} {LOCAL_PROJECT}

assert os.path.exists(f'{LOCAL_PROJECT}/backend/training/train_stage2.py'), "❌ Clone failed"
print(f"\n✅ Project ready: {LOCAL_PROJECT}")

## 🤗 1.4 — Tải SDXL Inpainting Model (tự động)

Tải `diffusers/stable-diffusion-xl-1.0-inpainting-0.1` từ HuggingFace.
Tải thẳng vào local mỗi lần Colab khởi động (~10 phút).

In [ ]:
import os

LOCAL_PROJECT = '/content/TryHairStyle'
LOCAL_SDXL = f'{LOCAL_PROJECT}/backend/models/stable-diffusion/sd_xl_inpainting'

required_dirs = ['vae', 'unet', 'tokenizer', 'tokenizer_2', 'text_encoder', 'text_encoder_2', 'scheduler']

def check_sdxl(path):
    return os.path.exists(path) and all(os.path.exists(f'{path}/{d}') for d in required_dirs)

if check_sdxl(LOCAL_SDXL):
    print('\u2705 SDXL model \u0111\u00e3 c\u00f3 tr\u00ean local')
else:
    print('\U0001f4e5 \u0110ang t\u1ea3i SDXL Inpainting model t\u1eeb HuggingFace \u2192 local...')
    print('   (~10 ph\u00fat, t\u1ea3i l\u1ea1i m\u1ed7i l\u1ea7n Colab reset)')
    
    !pip install -q huggingface_hub
    from huggingface_hub import snapshot_download
    
    os.makedirs(LOCAL_SDXL, exist_ok=True)
    snapshot_download(
        repo_id='diffusers/stable-diffusion-xl-1.0-inpainting-0.1',
        local_dir=LOCAL_SDXL,
        ignore_patterns=['*.md', '*.png', '.gitattributes'],
    )
    print(f'\u2705 SDXL model \u0111\u00e3 t\u1ea3i v\u1ec1: {LOCAL_SDXL}')

assert check_sdxl(LOCAL_SDXL), f'\u274c SDXL model incomplete at {LOCAL_SDXL}'
print('\u2705 SDXL model ready')


## 🔗 1.5 — Symlink Dataset Chunks + Checkpoints

In [ ]:
import os, glob, shutil

LOCAL_PROJECT = '/content/TryHairStyle'
LOCAL_TRAINING = f'{LOCAL_PROJECT}/backend/training'
DRIVE_DATA = '/content/drive/MyDrive/TryHairStyle'
DRIVE_CHECKPOINTS = '/content/drive/MyDrive/TryHairStyle/checkpoints'

def _link(src, dst):
    if os.path.islink(dst): os.remove(dst)
    if not os.path.exists(dst): os.symlink(src, dst)

# ── Dataset chunks ──
chunks = sorted(glob.glob(f'{DRIVE_DATA}/processed_*'))

print(f"📂 Found {len(chunks)} chunk(s) trên Drive")
for c in chunks:
    name = os.path.basename(c)
    _link(c, f'{LOCAL_TRAINING}/{name}')
    # Đếm samples
    meta = os.path.join(c, 'metadata.jsonl')
    n = sum(1 for l in open(meta) if l.strip()) if os.path.exists(meta) else '?'
    print(f"  🔗 {name}: {n} samples")

if not chunks:
    print("⚠️ KHÔNG TÌM THẤY DATASET!")
    print(f"   Upload các thư mục processed_NNN/ vào: {DRIVE_DATA}")

# ── Checkpoints → Drive ──
ckpt = f'{LOCAL_TRAINING}/checkpoints'
if os.path.islink(ckpt): os.remove(ckpt)
elif os.path.isdir(ckpt): shutil.rmtree(ckpt)
os.symlink(DRIVE_CHECKPOINTS, ckpt)
print(f"🔗 Checkpoints → Drive")

print("✅ Symlinks ready!")

## 📦 1.6 — Cài Dependencies

In [ ]:
!pip install -q \
    diffusers>=0.36.0 \
    transformers>=4.36.0 \
    accelerate>=1.0.0 \
    safetensors>=0.7.0 \
    bitsandbytes>=0.43.0 \
    insightface>=0.7.3 \
    facenet-pytorch>=2.6.0 \
    lpips>=0.1.4 \
    opencv-python-headless>=4.9.0 \
    scikit-image>=0.26.0 \
    easydict>=1.0 \
    deep-translator>=1.11.0

try:
    import xformers
    print(f"✅ xformers: {xformers.__version__}")
except ImportError:
    !pip install -q xformers

print("✅ Dependencies OK")

## 🔍 1.7 — Verify

In [ ]:
import os, glob

LOCAL_PROJECT = '/content/TryHairStyle'
LOCAL_TRAINING = f'{LOCAL_PROJECT}/backend/training'
errors = []

print("=" * 60)
print("🔍 VERIFY")
print("=" * 60)

# Scripts
for s in ['train_stage2.py', 'models/texture_encoder.py', 'export_model.py']:
    ok = os.path.exists(f'{LOCAL_TRAINING}/{s}')
    print(f"{'✅' if ok else '❌'} {s}")
    if not ok: errors.append(s)

# SDXL
sdxl = f'{LOCAL_PROJECT}/backend/models/stable-diffusion/sd_xl_inpainting'
for d in ['vae', 'unet', 'text_encoder', 'text_encoder_2']:
    ok = os.path.exists(f'{sdxl}/{d}')
    print(f"{'✅' if ok else '❌'} SDXL/{d}")
    if not ok: errors.append(f'SDXL/{d}')

# Chunks
chunks = sorted(glob.glob(f'{LOCAL_TRAINING}/processed_*'))
if not chunks:
    p = f'{LOCAL_TRAINING}/processed'
    if os.path.exists(p): chunks = [p]
total = 0
for c in chunks:
    meta = os.path.join(c, 'metadata.jsonl')
    if os.path.exists(meta):
        n = sum(1 for l in open(meta) if l.strip())
        total += n

print(f"\n{'='*60}")
if errors:
    print(f"⚠️ Missing: {errors}")
else:
    print(f"✅ ALL OK — {len(chunks)} chunk(s), {total:,} samples")
    print("🚀 Ready!")

## ⏱️ 1.8 — Anti-timeout

In [ ]:
import IPython
display(IPython.display.HTML('<script>setInterval(()=>{document.querySelectorAll("colab-connect-button").forEach(b=>b.click())},60000)</script>'))
print("✅ Auto-reconnect enabled")

---
# PHẦN 2: STAGE 1 — TEXTURE ENCODER
---

ResNet50 → Trích xuất đặc trưng chất liệu tóc. ~30 phút.

In [ ]:
import os, sys

LOCAL_PROJECT = '/content/TryHairStyle'
os.chdir(LOCAL_PROJECT)
sys.path.insert(0, LOCAL_PROJECT)
os.environ['COLAB_GPU'] = '1'

# ╔══════════════════════════════════════════╗
# ║  🔧 STAGE 1 PARAMETERS                  ║
# ╚══════════════════════════════════════════╝
S1_EPOCHS = 20
S1_BATCH = 16
S1_MAX_SAMPLES = 0    # 0=all, nhỏ hơn để test
S1_RESUME = True
# ══════════════════════════════════════════

cmd = f"python -u backend/training/models/texture_encoder.py --epochs {S1_EPOCHS} --batch-size {S1_BATCH} --max-samples {S1_MAX_SAMPLES}"
if S1_RESUME: cmd += " --resume"

print(f"🧬 STAGE 1: Texture Encoder")
print(f"🚀 {cmd}")
print("=" * 60)
!{cmd}

---
# PHẦN 3: STAGE 2 — HAIR INPAINTING
---

9-Channel SDXL UNet, Chunked Loading – Global Epoch. ~12-48h.

| Param | Default | Nghĩa |
|-------|---------|-------|
| `S2_EPOCHS` | 5 | Số global epochs |
| `S2_BATCH` | 1 | Batch (1 cho T4) |
| `S2_MAX_SAMPLES` | 0 | Samples/chunk (0=all) |
| `S2_ACCUMULATION` | 8 | Effective batch = 8 |

In [ ]:
import os, sys

LOCAL_PROJECT = '/content/TryHairStyle'
os.chdir(LOCAL_PROJECT)
sys.path.insert(0, LOCAL_PROJECT)
os.environ['COLAB_GPU'] = '1'

# ╔══════════════════════════════════════════╗
# ║  🔧 STAGE 2 PARAMETERS                  ║
# ╚══════════════════════════════════════════╝
S2_EPOCHS = 5
S2_BATCH = 1
S2_MAX_SAMPLES = 0    # 0=all, 50 để test
S2_RESOLUTION = 512
S2_ACCUMULATION = 8
S2_FRESH = False      # True = từ đầu
# ══════════════════════════════════════════

cmd = f"python -u backend/training/train_stage2.py --epochs {S2_EPOCHS} --batch-size {S2_BATCH} --max-samples-per-chunk {S2_MAX_SAMPLES} --resolution {S2_RESOLUTION} --accumulation {S2_ACCUMULATION}"
if S2_FRESH: cmd += " --fresh"

print(f"🎨 STAGE 2: Hair Inpainting")
print(f"🚀 {cmd}")
print("=" * 60)
!{cmd}

---
# PHẦN 4: STAGE 3 — EXPORT
---

In [ ]:
import os, sys
LOCAL_PROJECT = '/content/TryHairStyle'
os.chdir(LOCAL_PROJECT)
sys.path.insert(0, LOCAL_PROJECT)

print("📊 STAGE 3: Export Best Model")
print("=" * 60)
!python -u backend/training/export_model.py

---
# PHẦN 5: MONITORING
---

## 📊 Loss Chart

In [ ]:
from IPython.display import Image, display
import os

charts = '/content/TryHairStyle/backend/training/checkpoints/charts'
latest = f'{charts}/loss_chart_latest.png'
if os.path.exists(latest):
    display(Image(filename=latest, width=900))
else:
    print("⚠️ Chưa có chart. Chạy training trước.")

## 📈 Training History

In [ ]:
import json, os

hp = '/content/TryHairStyle/backend/training/checkpoints/training_history.json'
if os.path.exists(hp):
    h = json.load(open(hp))
    print(f"Steps: {len(h.get('total_loss',[]))} | Epochs: {len(h.get('epoch_avg_loss',[]))}")
    if h.get('epoch_avg_loss'):
        print(f"\n{'Ep':<5} {'Train':<14} {'Val':<14} {'LPIPS':<10}")
        print('-'*43)
        for i, t in enumerate(h['epoch_avg_loss']):
            v = h['val_loss'][i] if i < len(h.get('val_loss',[])) else 'N/A'
            l = h['val_lpips'][i] if i < len(h.get('val_lpips',[])) else 0
            print(f"{i+1:<5} {t:<14.6f} {(f'{v:.6f}' if isinstance(v,float) else v):<14} {(f'{l:.4f}' if l>0 else 'N/A'):<10}")
else:
    print("⚠️ Chưa có history.")

## 💾 Checkpoints

In [ ]:
import os
DRIVE_CHECKPOINTS = '/content/drive/MyDrive/TryHairStyle/checkpoints'
if os.path.exists(DRIVE_CHECKPOINTS):
    total = 0
    for f in sorted(os.listdir(DRIVE_CHECKPOINTS)):
        fp = os.path.join(DRIVE_CHECKPOINTS, f)
        if os.path.isfile(fp):
            s = os.path.getsize(fp); total += s
            u = f'{s/1024**3:.2f} GB' if s>1024**3 else f'{s/1024**2:.1f} MB' if s>1024**2 else f'{s/1024:.0f} KB'
            print(f"  {f:<45} {u}")
    print(f"  {'TOTAL':<45} {total/1024**3:.2f} GB")
else:
    print("⚠️ No checkpoints yet.")

## 🧹 Cleanup VRAM

In [ ]:
import gc, torch
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"🧹 Allocated={torch.cuda.memory_allocated()/1024**3:.2f}GB, Reserved={torch.cuda.memory_reserved()/1024**3:.2f}GB")
print("✅ Done")

---
# PHẦN 6: QUICK RESUME & TEST
---

## 🔄 QUICK RESUME (1 cell — sau disconnect)

Chạy **cell này duy nhất** khi Colab reconnect. Tự động:
1. Mount Drive
2. Clone/pull code
3. Tải/link SDXL model
4. Symlinks
5. Dependencies
6. Resume training

In [ ]:
# ╔══════════════════════════════════════════════╗
# ║  🔧 RESUME CONFIG                           ║
# ╚══════════════════════════════════════════════╝
RESUME_STAGE = 2                    # 1 hoặc 2
RESUME_EPOCHS = 5
RESUME_BATCH = 1
RESUME_MAX_SAMPLES_PER_CHUNK = 0
RESUME_ACCUMULATION = 8
RESUME_RESOLUTION = 512
# ══════════════════════════════════════════════

import os, sys, shutil, glob

# --- 1. Mount ---
from google.colab import drive
drive.mount('/content/drive')

GITHUB_REPO = 'https://github.com/Phatjhhoq8/TryHairstyle_Diffusion.git'
GITHUB_BRANCH = 'master'
DRIVE_DATA = '/content/drive/MyDrive/TryHairStyle'
DRIVE_CKPT = '/content/drive/MyDrive/TryHairStyle/checkpoints'
LP = '/content/TryHairStyle'
LT = f'{LP}/backend/training'

# --- 2. Clone/Pull ---
if os.path.exists(LP):
    !cd {LP} && git pull origin {GITHUB_BRANCH} 2>/dev/null || true
else:
    !git clone -b {GITHUB_BRANCH} --depth 1 {GITHUB_REPO} {LP}

# --- 3. SDXL ---\nsdxl_local = f'{LP}/backend/models/stable-diffusion/sd_xl_inpainting'
req = ['vae','unet','text_encoder','text_encoder_2','tokenizer','tokenizer_2','scheduler']
if not (os.path.exists(sdxl_local) and all(os.path.exists(f'{sdxl_local}/{d}') for d in req)):
    print('\U0001f4e5 Downloading SDXL model to local...')
    !pip install -q huggingface_hub
    from huggingface_hub import snapshot_download
    os.makedirs(sdxl_local, exist_ok=True)
    snapshot_download('diffusers/stable-diffusion-xl-1.0-inpainting-0.1', local_dir=sdxl_local, ignore_patterns=['*.md','*.png','.gitattributes'])
print('\u2705 SDXL ready')

# --- 4. Symlinks ---
def _lnk(s, d):
    if os.path.islink(d): os.remove(d)
    if not os.path.exists(d): os.symlink(s, d)

for c in sorted(glob.glob(f'{DRIVE_DATA}/processed_*')):
    _lnk(c, f'{LT}/{os.path.basename(c)}')

ckpt = f'{LT}/checkpoints'
if os.path.islink(ckpt): os.remove(ckpt)
elif os.path.isdir(ckpt): shutil.rmtree(ckpt)
os.makedirs(DRIVE_CKPT, exist_ok=True)
os.symlink(DRIVE_CKPT, ckpt)

# --- 5. Deps ---
!pip install -q diffusers>=0.36.0 transformers>=4.36.0 accelerate>=1.0.0 safetensors>=0.7.0 bitsandbytes>=0.43.0 insightface>=0.7.3 facenet-pytorch>=2.6.0 lpips>=0.1.4 opencv-python-headless>=4.9.0 scikit-image>=0.26.0 easydict>=1.0 deep-translator>=1.11.0 2>/dev/null

# --- 6. Run ---
os.chdir(LP)
sys.path.insert(0, LP)
os.environ['COLAB_GPU'] = '1'

print(f"\n🔄 Resuming Stage {RESUME_STAGE}...")
print("=" * 60)

if RESUME_STAGE == 1:
    !python -u backend/training/models/texture_encoder.py --epochs {RESUME_EPOCHS} --batch-size 16 --resume
else:
    !python -u backend/training/train_stage2.py --epochs {RESUME_EPOCHS} --batch-size {RESUME_BATCH} --max-samples-per-chunk {RESUME_MAX_SAMPLES_PER_CHUNK} --resolution {RESUME_RESOLUTION} --accumulation {RESUME_ACCUMULATION}

## 🧪 Smoke Test (test nhanh)

In [ ]:
import os, sys
LOCAL_PROJECT = '/content/TryHairStyle'
os.chdir(LOCAL_PROJECT)
sys.path.insert(0, LOCAL_PROJECT)
os.environ['COLAB_GPU'] = '1'

print("🧪 TEST: Stage 1 (50 samples, 2ep)")
!python -u backend/training/models/texture_encoder.py --epochs 2 --batch-size 8 --max-samples 50

print("\n🧪 TEST: Stage 2 (50 samples/chunk, 1ep)")
!python -u backend/training/train_stage2.py --epochs 1 --batch-size 1 --max-samples-per-chunk 50 --accumulation 4 --fresh

print("\n🧪 TEST: Stage 3")
!python -u backend/training/export_model.py

print("\n✅ SMOKE TEST OK")

## 🚀 Full Pipeline (Stage 1→2→3)

In [ ]:
import os, sys, time
LOCAL_PROJECT = '/content/TryHairStyle'
os.chdir(LOCAL_PROJECT)
sys.path.insert(0, LOCAL_PROJECT)
os.environ['COLAB_GPU'] = '1'

# ╔══════════════════════════════════════════╗
# ║  🔧 FULL PIPELINE CONFIG                ║
# ╚══════════════════════════════════════════╝
FP_S1_EPOCHS = 20
FP_S2_EPOCHS = 5
FP_S2_BATCH = 1
FP_S2_MAX_SAMPLES = 0
FP_S2_ACCUMULATION = 8
# ══════════════════════════════════════════

t0 = time.time()

print("\n" + "="*60 + "\n🧬 STAGE 1\n" + "="*60)
t1 = time.time()
!python -u backend/training/models/texture_encoder.py --epochs {FP_S1_EPOCHS} --batch-size 16 --resume
s1 = (time.time()-t1)/60

print("\n" + "="*60 + "\n🎨 STAGE 2\n" + "="*60)
t2 = time.time()
!python -u backend/training/train_stage2.py --epochs {FP_S2_EPOCHS} --batch-size {FP_S2_BATCH} --max-samples-per-chunk {FP_S2_MAX_SAMPLES} --accumulation {FP_S2_ACCUMULATION}
s2 = (time.time()-t2)/60

print("\n" + "="*60 + "\n📊 STAGE 3\n" + "="*60)
!python -u backend/training/export_model.py

tt = (time.time()-t0)/60
print(f"\n{'='*60}\n✅ DONE! S1={s1:.0f}min, S2={s2:.0f}min, Total={tt:.0f}min ({tt/60:.1f}h)")

---
## 📖 Hướng dẫn

| Trường hợp | Chạy cells |
|-----------|------------|
| **Lần đầu** | 1.1 → 1.8 → Stage 1 → Stage 2 → Stage 3 |
| **Sau disconnect** | **Quick Resume** (1 cell duy nhất) |
| **Test nhanh** | 1.1 → 1.8 → Smoke Test |
| **Full auto** | 1.1 → 1.8 → Full Pipeline |